# Multimodal Classification (Joint fusion technique)
This notebook demonstrates a multimodal crisis classification pipeline using a joint fusion approach. It combines text and image features for multitask learning on humanitarian and informativeness tasks.

## Imports and Setup
This section imports all required libraries for data handling, visualization, deep learning, and evaluation.
It also loads environment variables and sets up file paths for datasets and model checkpoints.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
import keras_hub


from itertools import cycle
from keras.applications import EfficientNetB0
from keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.optimizers import AdamW
from sklearn.calibration import calibration_curve
from sklearn.metrics import (auc, classification_report, confusion_matrix, roc_auc_score, roc_curve)
from sklearn.preprocessing import LabelEncoder, label_binarize
from tqdm import tqdm

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Load paths from .env file
dataset_dir = os.getenv("DATASET_DIR")
datasplits_dir = os.getenv("DATASPLITS_DIR")
models_dir = os.getenv("MODELS_DIR")

# Patgs to the data splits files
train_file = os.path.join(datasplits_dir, "mm_train.tsv")
val_file = os.path.join(datasplits_dir, "mm_val.tsv")
test_file = os.path.join(datasplits_dir, "mm_test.tsv")

# Load the data splits into pandas DataFrames
train_df = pd.read_csv(train_file, sep="\t")
val_df = pd.read_csv(val_file, sep="\t")
test_df = pd.read_csv(test_file, sep="\t")

In [ ]:
train_df.shape

In [ ]:
train_df.head(1)

In [ ]:
val_df.shape

In [ ]:
val_df.head(1)

In [ ]:
test_df.shape

In [ ]:
test_df.head(1)

## Labels Encoding
The categorical labels for both informativeness and humanitarian tasks are encoded into integer values using `LabelEncoder`.
This ensures compatibility with TensorFlow models and loss functions.

In [ ]:
# Encode categorical labels for both informativeness and humanitarian tasks
label_encoders = {}
for col in ["mm_info", "mm_human"]:
    le = LabelEncoder()
    # Fit the encoder on all values from train, val, and test splits
    all_values = pd.concat([train_df[col], val_df[col], test_df[col]])
    le.fit(all_values)
    # Transform each split in-place to integer labels
    train_df[col] = le.transform(train_df[col])
    val_df[col] = le.transform(val_df[col])
    test_df[col] = le.transform(test_df[col])
    # Store the encoder for later use (e.g., inverse transform)
    label_encoders[col] = le

## Datasets creation
Here, the notebook defines preprocessing functions for text and images, and builds TensorFlow datasets for training, validation, and testing.
Text is tokenized and images are preprocessed to match model input requirements.

In [ ]:
# Parameters
MAX_LENGTH = train_df["cleaned_tweet_text"].str.split().str.len().max()
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

In [ ]:
# Initialize text preprocessor for the RoBERTa model
text_preprocessor = keras_hub.models.RobertaTextClassifierPreprocessor.from_preset("roberta_base_en", sequence_length=MAX_LENGTH)

In [ ]:
def process_image(path):
    """
    Loads an image from a file path, decodes it, resizes it, and applies EfficientNet preprocessing.

    Args:
        path (tf.Tensor or str): Path to the image file.

    Returns:
        tf.Tensor: Preprocessed image tensor ready for model input.
    """
    # Read the image file from disk
    img = tf.io.read_file(path)
    # Decode the JPEG image to a tensor with 3 color channels (RGB)
    img = tf.image.decode_jpeg(img, channels=3)
    # Resize the image to the target IMAGE_SIZE
    img = tf.image.resize(img, IMAGE_SIZE)
    # Apply EfficientNet-specific preprocessing (scaling, normalization)
    img = efficientnet_preprocess(img)
    return img

def encode_example(text, img_path, info_label, human_label):
    """
    Encodes a single multimodal example for the dataset pipeline.

    This function processes the text and image inputs, applies the necessary preprocessing,
    and packages them together with their corresponding labels for use in a TensorFlow dataset.

    Args:
        text (str): The tweet text.
        img_path (str or tf.Tensor): Path to the associated image file.
        info_label (int or float): Informativeness label.
        human_label (int): Humanitarian category label.

    Returns:
        tuple: (inputs, labels) where
            inputs is a dict with keys:
                - "input_word_ids": tokenized text ids
                - "input_padding_mask": attention mask for text
                - "input_image": preprocessed image tensor
            labels is a dict with keys:
                - "info": informativeness label (float32)
                - "human": humanitarian label (int32)
    """
    # Preprocess the text using the RoBERTa text preprocessor
    text_out = text_preprocessor(text)  # returns token_ids & padding_mask
    input_word_ids = text_out["token_ids"]
    padding_mask = text_out["padding_mask"]

    # Preprocess the image using EfficientNet preprocessing
    image = process_image(img_path)

    # Prepare model input dictionary
    outputs = {
        "input_word_ids": input_word_ids,
        "input_padding_mask": padding_mask,
        "input_image": image
    }

    # Prepare label dictionary
    labels = {
        "info": tf.cast(info_label, tf.float32),
        "human": tf.cast(human_label, tf.int32)
    }
    return outputs, labels

def df_to_multimodal_dataset(df, batch_size=32, shuffle=True):
    """
    Converts a DataFrame into a TensorFlow multimodal dataset for model training or evaluation.

    This function extracts text, image paths, and labels from the DataFrame,
    applies preprocessing and encoding, and returns a batched, prefetched tf.data.Dataset.

    Args:
        df (pd.DataFrame): DataFrame containing columns for text, image paths, and labels.
        batch_size (int): Number of samples per batch.
        shuffle (bool): Whether to shuffle the dataset.

    Returns:
        tf.data.Dataset: A batched and prefetched dataset yielding (inputs, labels) tuples.
    """
    # Get absolute image paths
    img_paths = df["image_path"].apply(lambda p: os.path.normpath(os.path.join(dataset_dir, p))).values
    # Get cleaned tweet texts
    texts = df["cleaned_tweet_text"].values
    # Get informativeness labels
    infos = df["mm_info"].values
    # Get humanitarian category labels
    humans = df["mm_human"].values

    # Create a tf.data.Dataset from the extracted arrays
    ds = tf.data.Dataset.from_tensor_slices((texts, img_paths, infos, humans))
    # Map the encode_example function to preprocess and encode each sample
    ds = ds.map(encode_example, num_parallel_calls=tf.data.AUTOTUNE)
    # Shuffle the dataset if requested (useful for training)
    if shuffle:
        ds = ds.shuffle(buffer_size=700)
    # Batch and prefetch for performance
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [ ]:
train_ds = df_to_multimodal_dataset(train_df, batch_size=BATCH_SIZE, shuffle=True)
val_ds = df_to_multimodal_dataset(val_df, batch_size=BATCH_SIZE, shuffle=False)
test_ds = df_to_multimodal_dataset(test_df, batch_size=BATCH_SIZE, shuffle=False)

## Model Definition
This section defines the multimodal classifier architecture.
It includes separate branches for text (transformer-based) and image (CNN-based) features, a fusion mechanism, and multitask output heads for both informativeness and humanitarian classification.

In [ ]:
def build_multimodal_classifier(text_backbone, image_backbone):
    """
    Builds a multimodal classifier model that fuses text and image features for multitask learning.

    The model consists of:
      - A text branch using a transformer backbone (e.g., RoBERTa) to extract text features.
      - An image branch using a CNN backbone (e.g., EfficientNet) to extract image features.
      - A fusion mechanism that combines text and image features using a learned gating mechanism.
      - A shared MLP classifier head.
      - Two output heads:
          * "info": Binary classification for informativeness (sigmoid activation).
          * "human": Multiclass classification for humanitarian category (softmax activation).

    Args:
        text_backbone (keras.Model): Pretrained text backbone model.
        image_backbone (keras.Model): Pretrained image backbone model.

    Returns:
        keras.Model: A compiled multimodal multitask classification model.
    """
    # TEXT BRANCH
    input_word_ids = keras.Input(shape=(None,), dtype=tf.int32, name="input_word_ids")
    padding_mask = keras.Input(shape=(None,), dtype=tf.int32, name="input_padding_mask")

    text_output = text_backbone({"token_ids": input_word_ids, "padding_mask": padding_mask})
    text_features = text_output[:, 0, :]  # CLS token

    # IMAGE BRANCH
    image_input = keras.Input(shape=(*IMAGE_SIZE, 3), name="input_image")

    image_augmentation = keras.Sequential([
        keras.layers.RandomFlip("horizontal"),
        keras.layers.RandomRotation(0.1),
        keras.layers.RandomZoom(0.2)
    ])

    augmented_image = image_augmentation(image_input)
    image_output = image_backbone(augmented_image)
    image_features = keras.layers.GlobalAveragePooling2D()(image_output)

    # FUSION
    # projections
    text_proj = keras.layers.Dense(512, name="text_projection")(text_features)
    image_proj = keras.layers.Dense(512, name="image_projection")(image_features)
    # concatenate features and compute gate
    fusion_input = keras.layers.Concatenate()([text_proj, image_proj])
    λ = keras.layers.Dense(1, activation="sigmoid", name="modality_gate")(fusion_input)
    # modality gating fusion: (1-λ) * text + λ * image
    fused = keras.layers.Add(name="fused")([
        keras.layers.Multiply()([keras.layers.Lambda(lambda x: 1 - x)(λ), text_proj]),
        keras.layers.Multiply()([λ, image_proj])
    ])

    # CLASSIFIER MLP
    x = keras.layers.BatchNormalization()(fused)
    x = keras.layers.Dense(512)(x)
    x = keras.layers.Activation("relu")(x)
    x = keras.layers.Dropout(0.6)(x)

    # OUTPUT HEADS
    info_out = keras.layers.Dense(1, activation="sigmoid", name="info")(x)
    human_out = keras.layers.Dense(train_df["mm_human"].nunique(), activation="softmax", name="human")(x)

    # FINAL MODEL
    model = keras.Model(inputs={
        "input_word_ids": input_word_ids,
        "input_padding_mask": padding_mask,
        "input_image": image_input
    }, outputs={
        "info": info_out,
        "human": human_out
    })

    return model

In [ ]:
# Define backbone models (frozen at this stage)
text_backbone = keras_hub.models.RobertaBackbone.from_preset("roberta_base_en")
text_backbone.trainable = False

image_backbone = EfficientNetB0(include_top=False, weights="imagenet", input_shape=(224, 224, 3))
image_backbone.trainable = False

# Build full multimodal model
model = build_multimodal_classifier(text_backbone, image_backbone)

# Summary
model.summary()

## Training
The model is compiled and trained on the training dataset, with validation monitoring and early stopping.
Callbacks are used to save the best model weights and prevent overfitting.

In [ ]:
# Training hyperparameters
LR = 1e-4         # Learning rate
WD = 1e-5         # Weight decay (L2 regularization)
EPOCHS = 100      # Maximum number of epochs
PATIENCE = 5      # Early stopping patience

In [ ]:
# Compile
model.compile(
    optimizer=AdamW(learning_rate=LR, weight_decay=WD),
    loss={
        "info": "binary_crossentropy",
        "human": "sparse_categorical_crossentropy"
    },
    metrics={
        "info": "accuracy",
        "human": "accuracy"
    }
)

# Callbacks
callbacks = [
    ModelCheckpoint(f"{models_dir}/multimodal_trained.weights.h5", save_weights_only=True, save_best_only=True, monitor="val_loss"),
    EarlyStopping(patience=PATIENCE, restore_best_weights=True)
]

# Train
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

In [ ]:
def plot_training_history(history):
    """
    Plots training and validation metrics for each key in the Keras History object.

    Args:
        history (keras.callbacks.History): History object returned by model.fit().
    
    This function will create a separate plot for each metric (except those starting with 'val_'),
    showing both the training and validation curves for easy comparison.
    """
    for key in history.history:
        # Only plot metrics that are not validation metrics (those will be plotted together)
        if not key.startswith("val_"):
            plt.figure()
            plt.plot(history.history[key], label="train")
            plt.plot(history.history[f"val_{key}"], label="val")
            plt.title(key)
            plt.xlabel("Epoch")
            plt.ylabel("Value")
            plt.legend()
            plt.grid(True)
            plt.show()

In [ ]:
plot_training_history(history)

## Fine-tuning
After initial training, selected layers of the text and image backbones are unfrozen for fine-tuning.
This allows the model to adapt pretrained features to the specific crisis classification tasks.

In [ ]:
def unfreeze_text_backbone(text_backbone, last_n_layers=2):
    """
    Unfreezes the last N transformer layers of the text backbone for fine-tuning.

    Args:
        text_backbone (keras.Model): The text backbone model (e.g., RoBERTa).
        last_n_layers (int): Number of last transformer layers to unfreeze.

    Returns:
        list: Names of the layers that were unfrozen.
    """
    unfrozen = []
    for layer in text_backbone.layers:
        # Check if the layer is a transformer layer by name
        if layer.name.startswith("transformer_layer_"):
            layer_index = int(layer.name.split("_")[2])
            # Unfreeze only the last N transformer layers (except BatchNorm)
            if layer_index >= (12 - last_n_layers):
                if not isinstance(layer, keras.layers.BatchNormalization):
                    layer.trainable = True
                    unfrozen.append(layer.name)
        else:
            # Freeze all other layers
            layer.trainable = False
    return unfrozen

In [ ]:
def unfreeze_image_backbone(image_backbone, unfreeze_from="block6a"):
    """
    Unfreezes the image backbone from a specified block for fine-tuning.

    This function sets all layers starting from the specified block (by name prefix)
    to trainable, except for BatchNormalization layers, which remain frozen for stability.

    Args:
        image_backbone (keras.Model): The image backbone model (e.g., EfficientNet).
        unfreeze_from (str): Name prefix of the block from which to start unfreezing.

    Returns:
        list: Names of the layers that were unfrozen.
    """
    unfrozen = []
    unfreeze = False
    for layer in image_backbone.layers:
        # Start unfreezing when the layer name matches the specified block
        if layer.name.startswith(unfreeze_from):
            unfreeze = True
        # Unfreeze layers (except BatchNormalization) if in the unfreeze region
        if unfreeze and not isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = True
            unfrozen.append(layer.name)
        else:
            layer.trainable = False
    return unfrozen

In [ ]:
# Partial unfreeze
unfrozen_text_layers = unfreeze_text_backbone(text_backbone, last_n_layers=2)
unfrozen_image_layers = unfreeze_image_backbone(image_backbone, unfreeze_from="block6a")

# Log summary
print(f"Unfrozen TEXT layers ({len(unfrozen_text_layers)}):")
for l in unfrozen_text_layers: print(f"  {l}")

print(f"\nUnfrozen IMAGE layers ({len(unfrozen_image_layers)}):")
for l in unfrozen_image_layers: print(f"  {l}")

print(f"\nTotal trainable weights in model: {len(model.trainable_weights)}")

In [ ]:
# Fine-tuning hyperparameters
LR = 1e-6            # Lower learning rate for fine-tuning
WD = 1e-5            # Weight decay for fine-tuning
EPOCHS = 20          # Maximum epochs for fine-tuning
PATIENCE = 2         # Early stopping patience for fine-tuning

In [ ]:
# Compile
model.compile(
    optimizer=AdamW(learning_rate=LR, weight_decay=WD),
    loss={
        "info": "binary_crossentropy",
        "human": "sparse_categorical_crossentropy"
    },
    metrics={
        "info": "accuracy",
        "human": "accuracy"
    }
)

# Callbacks
callbacks = [
    ModelCheckpoint(f"{models_dir}/multimodal_fine_tuned.weights.h5", save_weights_only=True, save_best_only=True, monitor="val_loss"),
    EarlyStopping(patience=PATIENCE, restore_best_weights=True)
]

# Fine-tune
ft_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

In [ ]:
plot_training_history(ft_history)

## Evaluation
The trained model is evaluated on the test dataset.
Performance metrics, confusion matrices, ROC curves, and reliability diagrams are generated for both tasks to assess model quality.

In [ ]:
def plot_reliability_curve(y_true, y_prob, task_name):
    """
    Plots a reliability diagram (calibration curve) for predicted probabilities.

    Args:
        y_true (array-like): True binary or multiclass labels.
        y_prob (array-like): Predicted probabilities for the positive class or each class.
        task_name (str): Name of the task for plot title.
    """
    # Compute calibration curve (fraction of positives vs. mean predicted value)
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=10)
    plt.figure(figsize=(6, 5))
    plt.plot(prob_pred, prob_true, marker='o', label='Model')
    plt.plot([0, 1], [0, 1], linestyle='--', label='Perfectly calibrated')
    plt.title(f"{task_name} - Reliability Diagram")
    plt.xlabel("Predicted probability")
    plt.ylabel("True probability")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_roc_auc(y_true, y_scores, class_names, task_name):
	"""
	Plots ROC curve(s) and computes AUC for binary or multiclass classification.

	Args:
		y_true (array-like): True labels.
		y_scores (array-like): Predicted probabilities or scores.
		class_names (list): List of class names for multiclass.
		task_name (str): Name of the task for plot title.
	"""
	if len(np.unique(y_true)) == 2:
		# Binary classification ROC curve
		roc_auc = roc_auc_score(y_true, y_scores)
		fpr, tpr, _ = roc_curve(y_true, y_scores)

		plt.figure(figsize=(6, 5))
		plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.2f})")
		plt.plot([0, 1], [0, 1], "k--")
		plt.xlabel("False Positive Rate")
		plt.ylabel("True Positive Rate")
		plt.title(f"{task_name} - ROC Curve")
		plt.legend(loc="lower right")
		plt.grid(True)
		plt.tight_layout()
		plt.show()

	else:
		# Multiclass ROC curve (one-vs-rest for each class)
		n_classes = len(class_names)
		y_true_bin = label_binarize(y_true, classes=list(range(n_classes)))

		fpr = {}
		tpr = {}
		roc_auc = {}

		for i in range(n_classes):
			fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_scores[:, i])
			roc_auc[i] = auc(fpr[i], tpr[i])

		colors = cycle(["aqua", "darkorange", "cornflowerblue", "green", "red"])
		plt.figure(figsize=(8, 6))

		for i, color in zip(range(n_classes), colors):
			plt.plot(fpr[i], tpr[i], color=color, lw=2,
					 label=f"Class {class_names[i]} (AUC = {roc_auc[i]:.2f})")
		
		plt.plot([0, 1], [0, 1], "k--", lw=2)
		plt.xlim([0.0, 1.0])
		plt.ylim([0.0, 1.05])
		plt.xlabel("False Positive Rate")
		plt.ylabel("True Positive Rate")
		plt.title(f"{task_name} - ROC Curve")
		plt.legend(loc="lower right")
		plt.grid(True)
		plt.tight_layout()
		plt.show()

In [ ]:
def evaluate_task(y_true, y_pred, task_name, class_names=None, y_scores=None):
    """
    Evaluates classification performance for a given task, printing a classification report,
    plotting a confusion matrix, and (optionally) ROC-AUC and reliability diagrams.

    Args:
        y_true (array-like): True labels.
        y_pred (array-like): Predicted labels.
        task_name (str): Name of the task (for plot/report titles).
        class_names (list, optional): List of class names for display.
        y_scores (array-like, optional): Predicted probabilities or scores for ROC/reliability plots.
    """
    # Print the classification report with precision, recall, f1-score, and support
    print(f"\n{task_name} - Classification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    # Prepare display names for confusion matrix (capitalize for Task #2)
    if task_name == "Task #2":
        display_names = [c[0].upper() for c in class_names]
    else:
        display_names = class_names

    # Compute and plot the confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
                xticklabels=display_names, yticklabels=display_names)
    plt.title(f"{task_name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()
 
    # If probability scores are provided, plot ROC-AUC and reliability diagrams
    if y_scores is not None:
        plot_roc_auc(y_true, y_scores, class_names, task_name)
        # Only plot reliability diagram for binary tasks
        if len(np.unique(y_true)) == 2:
            plot_reliability_curve(y_true, y_scores, task_name)

In [ ]:
def evaluate_model(model, test_ds):
    """
    Runs inference on the test dataset and evaluates the model on both tasks.

    Args:
        model (tf.keras.Model): Trained multitask model.
        test_ds (tf.data.Dataset): Test dataset.

    This function collects predictions and true labels for both tasks:
      - Task 1: informativeness (binary classification)
      - Task 2: humanitarian category (multiclass classification)
    It then calls evaluate_task() for each task to print metrics and plots.
    """
    # Initialize lists to collect true labels, predicted labels, and predicted probabilities for both tasks
    y_true_info, y_pred_info, y_scores_info = [], [], []
    y_true_human, y_pred_human, y_scores_human = [], [], []
    
    # Iterate over the test dataset in batches
    for img_batch, label_batch in tqdm(test_ds, desc="Running inference on test set"):
        # Predict outputs for each task
        preds = model.predict(img_batch, verbose=0)

        # Task 1: informativeness (binary)
        info_probs = preds[0].flatten()  # Predicted probabilities for binary task
        y_scores_info.extend(info_probs)
        y_true_info.extend(label_batch["info"].numpy())
        y_pred_info.extend((preds[0] > 0.5).astype("int32").flatten())  # Threshold at 0.5

        # Task 2: humanitarian category (multiclass)
        human_probs = preds[1]  # Predicted probabilities for each class
        y_scores_human.extend(human_probs)
        y_true_human.extend(label_batch["human"].numpy())
        y_pred_human.extend(np.argmax(preds[1], axis=1))  # Predicted class index

    # Evaluate Task 1: informativeness
    evaluate_task(
        y_true_info,
        y_pred_info,
        "Task #1",
        class_names=["Informative", "Not Informative"],
        y_scores=np.array(y_scores_info)
    )

    # Evaluate Task 2: humanitarian category
    evaluate_task(
        y_true_human,
        y_pred_human,
        "Task #2",
        class_names=[
            "affected_individuals",
            "infrastructure_and_utility_damage",
            "not_humanitarian",
            "other_relevant_information",
            "rescue_volunteering_or_donation_effort"
        ],
        y_scores=np.array(y_scores_human)
    )

In [ ]:
# Load the best fine-tuned weights for the multimodal model
model.load_weights(f"{models_dir}/multimodal_fine_tuned.weights.h5")

# Evaluate the model on the test dataset and print metrics/plots for both tasks
evaluate_model(model, test_ds)

## Save model
The final fine-tuned model is saved in Keras format for future inference or deployment.

In [ ]:
model.save("./models/multimodal_fine_tuned.model.keras")